In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
silver_table = f"{catalog}.{schema}.silver_orders"
gold_table = f"{catalog}.{schema}.gold_sales_by_category_city"

silver_df = spark.table(silver_table)

gold_df = (
    silver_df
    .groupBy("category", "city")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("units_sold"),
        F.round(F.sum("total_amount"), 2).alias("total_sales"),
        F.round(F.avg("total_amount"), 2).alias("average_order_value")
    )
    .orderBy("category", "city")
)

(gold_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table))

print(f"Gold aggregation complete: {gold_table}")
display(spark.table(gold_table))